# 02_data_preprocessing

Extracts raw band statistics from Planet satellite images at each CTD sampling point.

**Input:**
- `geospatial/ctd_points.geojson` — 13 CTD sampling locations
- `data/Satellite/*.composite.tif` — 8-band Planet SuperDove composites

**Output:**
- `data/datasets/raw_features.parquet` — raw band statistics per CTD per date

In [11]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask
from tqdm import tqdm

## Configuration

In [12]:
# Paths
CTD_GEOJSON   = Path("../geospatial/ctd_points.geojson")
SATELLITE_DIR = Path("../data/Satellite")
OUTPUT_DIR    = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Buffer radius around each CTD point (metres, projected CRS)
BUFFER_M = 200

# Minimum valid pixels required inside a buffer to accept a sample
MIN_PIXELS = 50

# Planet SuperDove 8-band order
BAND_NAMES = ["coastal", "blue", "green_i", "green", "yellow", "red", "red_edge", "nir"]

# Statistics computed per band
STAT_NAMES = ["mean", "median", "std", "p10", "p90"]

## Load CTD sampling points

In [13]:
ctd = gpd.read_file(CTD_GEOJSON).set_crs("EPSG:4326")
print(f"CTD points loaded: {len(ctd)}")
ctd.head()

CTD points loaded: 12


,ControlPointCode,geometry
0,CTD1,POINT (-0.78448 37.8118)
1,CTD2,POINT (-0.8078 37.76062)
2,CTD3,POINT (-0.78355 37.76178)
3,CTD4,POINT (-0.74962 37.74823)
4,CTD5,POINT (-0.72712 37.74045)


## Discover satellite images

In [14]:
tif_files = sorted(SATELLITE_DIR.glob("*composite.tif"))

# Keep only 8-band images
valid_tifs = []
for f in tif_files:
    try:
        with rasterio.open(f) as src:
            if src.count == 8:
                valid_tifs.append(f)
    except Exception:
        pass

print(f"Total TIF files found : {len(tif_files)}")
print(f"Valid 8-band images   : {len(valid_tifs)}")

Total TIF files found : 56
Valid 8-band images   : 56


## Feature extraction

For each valid image × CTD combination, a circular buffer of `BUFFER_M` metres is
projected into the image CRS. All pixels inside the buffer are collected and five
summary statistics (mean, median, std, p10, p90) are computed **per band**.

Spectral indices (NDTI, band ratios, etc.) are computed downstream in
`04_feature_engineering.ipynb` so that their construction can be iterated without
re-running the expensive pixel extraction.

In [15]:
def extract_band_stats(
    image: np.ndarray,
    mask: np.ndarray,
    band_names: list,
    nodata=None,
    min_pixels: int = 50,
) -> dict | None:
    """
    Compute per-band summary statistics for pixels inside `mask`.

    Parameters
    ----------
    image      : (nbands, height, width) float32 array
    mask       : (height, width) boolean array — True = inside buffer
    band_names : list of band name strings
    nodata     : scalar nodata value to exclude (or None)
    min_pixels : minimum valid pixel count; returns None if below threshold

    Returns
    -------
    dict with keys like 'coastal_mean', 'blue_std', ..., 'n_pixels'
    or None if there are not enough valid pixels.
    """
    pixels = image[:, mask].T  # (n_pixels, nbands)

    if pixels.size == 0:
        return None

    # Remove nodata / NaN rows
    if nodata is not None:
        valid = ~np.any(pixels == nodata, axis=1)
    else:
        valid = ~np.any(np.isnan(pixels), axis=1)
    pixels = pixels[valid]

    if pixels.shape[0] < min_pixels:
        return None

    stats = {"n_pixels": pixels.shape[0]}
    for b, name in enumerate(band_names):
        vals = pixels[:, b]
        stats[f"{name}_mean"]   = float(vals.mean())
        stats[f"{name}_median"] = float(np.median(vals))
        stats[f"{name}_std"]    = float(vals.std())
        stats[f"{name}_p10"]    = float(np.percentile(vals, 10))
        stats[f"{name}_p90"]    = float(np.percentile(vals, 90))

    return stats

## Main extraction loop

In [16]:
records = []

for tif_path in tqdm(valid_tifs, desc="Processing images"):
    date_match = re.search(r"(\d{4}-\d{2}-\d{2})", tif_path.name)
    if not date_match:
        continue
    date_str = date_match.group(1)

    with rasterio.open(tif_path) as src:
        raster_crs = src.crs
        nodata     = src.nodata
        image      = src.read().astype("float32")

        # Reproject CTD points and create buffers in raster CRS
        ctd_proj    = ctd.to_crs(raster_crs)
        ctd_buffers = ctd_proj.copy()
        ctd_buffers["geometry"] = ctd_buffers.buffer(BUFFER_M)

        for _, row in ctd_buffers.iterrows():
            mask = geometry_mask(
                [row.geometry],
                transform=src.transform,
                invert=True,
                out_shape=(src.height, src.width),
            )

            stats = extract_band_stats(
                image,
                mask,
                band_names=BAND_NAMES,
                nodata=nodata,
                min_pixels=MIN_PIXELS,
            )

            if stats is None:
                continue

            record = {"ctd": row["ControlPointCode"], "date": date_str}
            record.update(stats)
            records.append(record)

print(f"\nTotal samples extracted: {len(records)}")

Processing images: 100%|██████████| 56/56 [01:50<00:00,  1.98s/it]


Total samples extracted: 597


## Build and save dataset

In [17]:
df = pd.DataFrame(records)
df["date"] = pd.to_datetime(df["date"]).dt.date
df["ctd"]  = df["ctd"].astype(str)

# Canonical column order: metadata first, then band features
feature_cols = [f"{b}_{s}" for b in BAND_NAMES for s in STAT_NAMES]
df = df[["ctd", "date", "n_pixels"] + feature_cols]

print(f"Dataset shape : {df.shape}")
print(f"Date range    : {df['date'].min()} → {df['date'].max()}")
print(f"CTD stations  : {sorted(df['ctd'].unique())}")
df.head()

Dataset shape : (597, 43)
Date range    : 2021-08-24 → 2024-07-03
CTD stations  : ['CTD1', 'CTD10', 'CTD11', 'CTD12', 'CTD2', 'CTD3', 'CTD4', 'CTD5', 'CTD6', 'CTD7', 'CTD8', 'CTD9']


,ctd,date,n_pixels,coastal_mean,coastal_median,coastal_std,coastal_p10,coastal_p90,blue_mean,blue_median,...,red_edge_mean,red_edge_median,red_edge_std,red_edge_p10,red_edge_p90,nir_mean,nir_median,nir_std,nir_p10,nir_p90
0,CTD1,2021-08-24,13943,8687.416992,8108.0,1320.282227,7559.0,10858.599609,7893.073730,7072.0,...,4359.936035,3733.0,1444.534546,3141.0,6741.000000,3181.511475,2804.0,1142.426880,2122.199951,5009.0
1,CTD1,2021-08-24,13943,8900.343750,8574.0,1232.989136,7707.0,11054.799805,8184.126953,7890.0,...,4919.408691,4605.0,1355.975830,3549.0,7152.600098,3640.087158,3375.0,1117.906616,2506.000000,5554.0
2,CTD2,2021-08-24,13936,7115.936523,7109.0,146.661041,6935.0,7311.000000,6241.397461,6212.0,...,2800.867920,2784.0,137.505142,2642.0,2993.500000,1878.186768,1866.0,123.503250,1733.000000,2046.0
3,CTD3,2021-08-24,13942,6726.054688,6727.0,99.009186,6605.0,6848.000000,5826.857422,5826.0,...,2491.512207,2490.0,73.885811,2407.0,2576.000000,1602.847778,1603.0,85.902649,1509.000000,1698.0
4,CTD4,2021-08-24,13937,10280.360352,10543.0,1228.415649,8343.0,11692.000000,9590.727539,9840.0,...,5833.093262,6077.0,1174.447754,3976.0,7197.000000,4151.604004,4332.0,871.784485,2787.000000,5151.0


In [18]:
out_path = OUTPUT_DIR / "raw_features.parquet"
df.to_parquet(out_path, index=False)
print(f"Saved → {out_path}")

Saved → ../data/processed/raw_features.parquet
